# Check the 2025 Industry share

This checks the **82.9%** shown in Chart 2, using Epoch AI’s original `all_ai_models.csv`.

Source: [Epoch AI — Data on AI models](https://epoch.ai/data/ai-models).

## 1. Read the CSV

In [1]:
import csv
from pathlib import Path
import pandas as pd

# Find the file from the project folder or this notebook's folder.
root = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "data/raw/ai_models/all_ai_models.csv").exists()
)

source = root / "data/raw/ai_models/all_ai_models.csv"
with source.open(encoding="utf-8-sig", newline="") as file:
    models = list(csv.DictReader(file))

print("Source:", source.name)
print("Rows in the file:", len(models))

Source: all_ai_models.csv
Rows in the file: 3628


## 2. Keep models published in 2025

The denominator includes every sector, including Other and Unknown.

In [2]:
models_2025 = [row for row in models
               if row["Publication date"].startswith("2025-")]
total = len(models_2025)
print("Total models in 2025:", total)

pd.DataFrame(models_2025)[
    ["Model", "Publication date", "Organization categorization"]
].head()

Total models in 2025: 578


,Model,Publication date,Organization categorization
0,Solar Open 100B,2025-12-31,Industry
1,K-EXAONE,2025-12-31,Industry
2,VAETKI,2025-12-30,Industry
3,A.X K1,2025-12-30,Industry
4,HyperCLOVA X SEED 32B Think,2025-12-29,Industry


## 3. Count Industry models

Count Industry-only tags. Repeated tags such as `Industry,Industry` still count as Industry. Models tagged with both Industry and Academia are counted separately as Collaboration.

In [3]:
def industry_only(value):
    tags = {tag.strip() for tag in value.split(",") if tag.strip()}
    return tags == {"Industry"}

industry_models = [row for row in models_2025
                   if industry_only(row["Organization categorization"])]
industry_count = len(industry_models)
print("Industry models in 2025:", industry_count)

pd.DataFrame(industry_models)[
    ["Model", "Publication date", "Organization categorization"]
].head()

Industry models in 2025: 479


,Model,Publication date,Organization categorization
0,Solar Open 100B,2025-12-31,Industry
1,K-EXAONE,2025-12-31,Industry
2,VAETKI,2025-12-30,Industry
3,A.X K1,2025-12-30,Industry
4,HyperCLOVA X SEED 32B Think,2025-12-29,Industry


## 4. Calculate the percentage

In [4]:
share = industry_count / total * 100
print(f"{industry_count} / {total} × 100 = {share:.5f}%")
print(f"Rounded to one decimal place: {share:.1f}%")

479 / 578 × 100 = 82.87197%
Rounded to one decimal place: 82.9%


## 5. Compare with the dashboard

Use the dashboard’s processing functions to check the two counts and the rounded percentage.

In [5]:
import sys
sys.path.insert(0, str(root.resolve()))
from src.preprocessing import load_models, aggregate_counts, calculate_shares

app_models, _ = load_models(source)
app_shares = calculate_shares(aggregate_counts(app_models, 2025, 2025))
point = app_shares.loc[app_shares["Sector"] == "Industry"].iloc[0]

assert industry_count == point["Count"]
assert total == point["Total"]
assert round(share, 1) == round(point["Share"], 1)
print(f"Matches the dashboard: {int(point['Count'])} / {int(point['Total'])} = {point['Share']:.1f}%")

Matches the dashboard: 479 / 578 = 82.9%
